# Multi-Modal DistilBERT Yelp Fake Detector — PURE FULL (642k, no AI)

End-to-end **DistilBERT + behavioural** fusion model — the transformer member of Model B:
* **Text branch:** DistilBERT encodes the review → [CLS] embedding (768-d)
* **Behavioural branch:** MLP over the 17 reviewer/business features
* **Fusion head:** concat(text, behavioural) → dense → 1 logit, trained **end-to-end**

**This variant trains on the PURE full Yelp dataset** (`yelp_multimodal_features_full.csv`,
678k reviews, deduped YelpZip+NYC+Chi) — **no AI reviews**, the true human-fake task. The train
split is the real **~11% fake** distribution (val/test balanced 50/50), so we handle the
imbalance with a **class `pos_weight`** in the loss (no AI ×10) + a **val-tuned threshold**.
Early stopping on **val macro-F1**. Deep counterpart to the full pure sklearn Model B
(ens 76.2% / xgb 77.3%).

**Before running:** upload `data/yelp_multimodal_features_full.csv` (~480 MB) to Drive root;
**GPU** runtime (high-RAM recommended — 642k rows; a strong GPU like A100/L4 is advisable, this
is a large fine-tune). **Output:** downloads `yelp_multimodal_distilbert_full_onnx.zip` (ONNX +
scaler + meta + tokenizer/) → extract to `data/yelp_multimodal_distilbert_full_onnx/`. Feature
order matches `model_b.FEAT_COLS`, so it is serving-compatible. Reports macro-F1 + fake
recall/precision + genuine specificity.

In [ ]:
!pip install -q -U transformers

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizerFast
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, classification_report
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')

CSV='/content/drive/MyDrive/yelp_multimodal_features_full.csv'   # PURE full 678k (no AI)
MAX_LEN=256; BATCH=64; EPOCHS=4; LR=2e-5     # BATCH 64 to cut steps on 642k; drop to 32 if OOM
DROP={'user_reviews_on_this_biz'}
META={'text','label','split','source','fake_type','generator','length_bucket'}
device='cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

df=pd.read_csv(CSV); df['text']=df['text'].astype(str)
feat_cols=[c for c in df.columns if c not in META and c not in DROP]
print('rows', len(df), '| behavioural features', len(feat_cols))
tr=df[df.split=='train'].reset_index(drop=True)
va=df[df.split=='val'].reset_index(drop=True)
te=df[df.split=='test'].reset_index(drop=True)

sc=StandardScaler()
Btr=sc.fit_transform(tr[feat_cols].values).astype('float32')
Bva=sc.transform(va[feat_cols].values).astype('float32')
Bte=sc.transform(te[feat_cols].values).astype('float32')

# CLASS IMBALANCE (pure full data): train is the true ~11% fake (val/test balanced). No AI ×10 —
# instead per-sample weight = n_neg/n_pos for the minority (== BCE pos_weight); val-tuned thr sets
# the final operating point.
POS_WEIGHT=float((tr.label.values==0).sum())/max(int((tr.label.values==1).sum()),1)
sw_tr=np.where(tr.label.values==1, POS_WEIGHT, 1.0).astype('float32')
print('train fake rate', round(float(tr.label.mean()),3), '| POS_WEIGHT', round(POS_WEIGHT,2))

In [ ]:
tok=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class MMData(Dataset):
    def __init__(self, texts, beh, labels, weights=None):
        self.enc=tok(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        self.beh=torch.tensor(beh); self.y=torch.tensor(np.asarray(labels), dtype=torch.float32)
        self.w=torch.tensor(weights if weights is not None else np.ones(len(labels), dtype='float32'))
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (self.enc['input_ids'][i], self.enc['attention_mask'][i],
                self.beh[i], self.y[i], self.w[i])

tr_dl=DataLoader(MMData(tr.text.values, Btr, tr.label.values, sw_tr), batch_size=BATCH, shuffle=True)
va_dl=DataLoader(MMData(va.text.values, Bva, va.label.values), batch_size=64)
te_dl=DataLoader(MMData(te.text.values, Bte, te.label.values), batch_size=64)

In [ ]:
class MultiModalDistilBert(nn.Module):
    def __init__(self, n_beh):
        super().__init__()
        self.bert=DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.text_proj=nn.Sequential(nn.Linear(768,256), nn.ReLU(), nn.Dropout(0.3))
        self.beh=nn.Sequential(nn.Linear(n_beh,64), nn.ReLU(), nn.Dropout(0.3),
                               nn.Linear(64,32), nn.ReLU())
        self.head=nn.Sequential(nn.Linear(256+32,64), nn.ReLU(), nn.Dropout(0.4), nn.Linear(64,1))
    def forward(self, ids, mask, beh):
        cls=self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:,0]   # [CLS]
        t=self.text_proj(cls); b=self.beh(beh)
        return self.head(torch.cat([t,b], dim=1)).squeeze(-1)

model=MultiModalDistilBert(len(feat_cols)).to(device)

# Layer-wise LR decay (LLRD): deeper DistilBERT layers + new heads train faster,
# embeddings slowest — the project's standard fine-tuning recipe.
def make_optimizer(m, base_lr=LR, head_lr=1e-3, decay=0.95, wd=0.01):
    b=m.bert; n=len(b.transformer.layer); g=[]
    g.append({'params':list(b.embeddings.parameters()), 'lr':base_lr*(decay**n)})
    for i,layer in enumerate(b.transformer.layer):
        g.append({'params':list(layer.parameters()), 'lr':base_lr*(decay**(n-1-i))})
    g.append({'params':list(m.text_proj.parameters())+list(m.beh.parameters())+list(m.head.parameters()),
              'lr':head_lr})
    return torch.optim.AdamW(g, weight_decay=wd)
opt=make_optimizer(model)
loss_fn=nn.BCEWithLogitsLoss(reduction='none')
print('params:', sum(p.numel() for p in model.parameters())/1e6, 'M')

In [ ]:
USE_AMP = (device == 'cuda')          # mixed precision on GPU (faster), plain on CPU

def predict(dl):
    model.eval(); probs=[]
    with torch.no_grad():
        for ids,mask,beh,y,w in dl:
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logit=model(ids.to(device), mask.to(device), beh.to(device))
            else:
                logit=model(ids.to(device), mask.to(device), beh.to(device))
            probs.append(torch.sigmoid(logit.float()).cpu().numpy())
    return np.concatenate(probs)

def best_thr(y,p):
    bf,bt=-1,0.5
    for t in np.linspace(0.05,0.95,19):
        s=f1_score(y,(p>=t).astype(int),average='macro')
        if s>bf: bf,bt=s,t
    return bt,bf

total_steps=len(tr_dl)*EPOCHS
scheduler=get_linear_schedule_with_warmup(opt, int(0.1*total_steps), total_steps)   # 10% warmup
scaler=torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_f1, best_state, best_thr_v, wait, PATIENCE = -1, None, 0.5, 0, 3
tr_hist, val_hist = [], []
for ep in range(EPOCHS):
    model.train(); tot=0
    pbar=tqdm(tr_dl, desc=f'epoch {ep+1}/{EPOCHS}', leave=False)
    for ids,mask,beh,y,w in pbar:
        opt.zero_grad()
        if USE_AMP:
            with torch.amp.autocast('cuda'):
                logit=model(ids.to(device), mask.to(device), beh.to(device))
                loss=(loss_fn(logit, y.to(device))*w.to(device)).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            logit=model(ids.to(device), mask.to(device), beh.to(device))
            loss=(loss_fn(logit, y.to(device))*w.to(device)).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step(); tot+=loss.item()
        pbar.set_postfix(loss=f'{loss.item():.3f}', lr=f'{scheduler.get_last_lr()[-1]:.1e}')
    p_va=predict(va_dl); thr,f1=best_thr(va.label.values, p_va)
    tr_hist.append(tot/len(tr_dl)); val_hist.append(f1)
    print(f'epoch {ep+1}: train_loss={tot/len(tr_dl):.4f}  val_macroF1={f1:.4f} @thr {thr:.2f}')
    if f1>best_f1:
        best_f1, best_thr_v = f1, thr
        best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait+=1
        if wait>=PATIENCE: print('  early stopping'); break
model.load_state_dict(best_state); print(f'restored best (val macroF1={best_f1:.4f}, thr={best_thr_v:.2f})')

In [ ]:
# Training curves
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(1,len(tr_hist)+1),tr_hist,marker='o'); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(range(1,len(val_hist)+1),val_hist,marker='o',color='green'); ax[1].set_title('Val macro-F1'); ax[1].set_xlabel('epoch')
ax[1].axvline(int(np.argmax(val_hist))+1, ls='--', color='red', label='best epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('yelp_multimodal_distilbert_training.png', dpi=120); plt.show()

In [ ]:
p_te=predict(te_dl); pred=(p_te>=best_thr_v).astype(int); y_te=te.label.values
print('threshold =', round(best_thr_v,2))
print('test accuracy  =', round(accuracy_score(y_te,pred),4))
print('test macro-F1  =', round(f1_score(y_te,pred,average='macro'),4))
print('fake recall    =', round(recall_score(y_te,pred,pos_label=1),3))
print('fake precision =', round(precision_score(y_te,pred,pos_label=1),3))
print('genuine spec   =', round(recall_score(y_te,pred,pos_label=0),3))
print('\nReferences (same Yelp test): text-only ~69% · pure sklearn Model B (full) ens 76.2% / xgb 77.3%')
print()
print(classification_report(y_te, pred, target_names=['genuine','fake'], digits=4))

In [ ]:
# Export to ONNX (thread-safe serving) + bundle scaler/threshold/tokenizer
!pip install -q onnx onnxruntime onnxscript
import os, json, joblib, torch, zipfile, numpy as np
model.eval()
dummy_ids =torch.zeros(1,MAX_LEN,dtype=torch.long,device=device)
dummy_mask=torch.ones(1,MAX_LEN,dtype=torch.long,device=device)
dummy_beh =torch.zeros(1,len(feat_cols),dtype=torch.float32,device=device)
_args=(dummy_ids,dummy_mask,dummy_beh)
_kw=dict(input_names=['input_ids','attention_mask','beh'], output_names=['logit'],
         dynamic_axes={'input_ids':{0:'b'},'attention_mask':{0:'b'},'beh':{0:'b'},'logit':{0:'b'}},
         opset_version=14, do_constant_folding=True)
try:
    torch.onnx.export(model,_args,'yelp_multimodal_distilbert.onnx',dynamo=False,**_kw)  # stable legacy exporter
except TypeError:
    torch.onnx.export(model,_args,'yelp_multimodal_distilbert.onnx',**_kw)               # older torch: legacy is default
joblib.dump(sc,'yelp_multimodal_distilbert_scaler.joblib')
tok.save_pretrained('yelp_multimodal_distilbert_tok')   # HF tokenizer = a folder of files
json.dump({'threshold':float(best_thr_v),'feat_cols':feat_cols,'max_len':MAX_LEN},
          open('yelp_multimodal_distilbert_meta.json','w'), indent=2)

# sanity check: ONNX vs PyTorch on 4 test rows
import onnxruntime as ort
sess=ort.InferenceSession('yelp_multimodal_distilbert.onnx',providers=['CPUExecutionProvider'])
enc=tok(list(te.text.values[:4]),truncation=True,padding='max_length',max_length=MAX_LEN,return_tensors='np')
onnx_logit=sess.run(None,{'input_ids':enc['input_ids'].astype('int64'),
                          'attention_mask':enc['attention_mask'].astype('int64'),
                          'beh':Bte[:4]})[0].flatten()
print('ONNX inputs:', [i.name for i in sess.get_inputs()])
print('ONNX prob :', np.round(1/(1+np.exp(-onnx_logit)),4))
print('Torch prob:', np.round(p_te[:4],4))

with zipfile.ZipFile('yelp_multimodal_distilbert_full_onnx.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['yelp_multimodal_distilbert.onnx','yelp_multimodal_distilbert_scaler.joblib',
               'yelp_multimodal_distilbert_meta.json']:
        zf.write(fn)
    for f in os.listdir('yelp_multimodal_distilbert_tok'):   # bundle the tokenizer folder INSIDE the one zip
        zf.write(os.path.join('yelp_multimodal_distilbert_tok', f), arcname=f'tokenizer/{f}')
from google.colab import files
files.download('yelp_multimodal_distilbert_full_onnx.zip')  # extract -> data/yelp_multimodal_distilbert_full_onnx/
print('done: one zip -> onnx + scaler + meta + tokenizer/  (extract -> data/yelp_multimodal_distilbert_full_onnx/)')